In [1]:
! rm -rf /kaggle/working/TransPatch

In [2]:
%cd /kaggle/working/

/kaggle/working


In [3]:
%ls -a 

./  ../  __notebook__.ipynb


In [4]:
!git clone --branch ablations --single-branch https://github.com/AkshatT2307/TransPatch.git


Cloning into 'TransPatch'...
remote: Enumerating objects: 186, done.
remote: Counting objects: 100% (186/186), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 186 (delta 78), reused 154 (delta 46), pack-reused 0 (from 0)
Receiving objects: 100% (186/186), 7.67 MiB | 33.86 MiB/s, done.
Resolving deltas: 100% (78/78), done.


In [5]:
%cd /kaggle/working/TransPatch

/kaggle/working/TransPatch


In [6]:
# !git pull

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import sys
sys.path.append('/kaggle/working/TransPatch')
from utils.utils import setup_logging, get_config_from_yaml, process_config, print_config
# from trainer.trainer_sidewalk import PatchTrainer
from trainer.trainer_vit2cnn import PatchTrainer
import torch
import pickle

2026-01-24 18:46:20.858012: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769280381.047735      24 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769280381.101954      24 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769280381.570078      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769280381.570118      24 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769280381.570121      24 computation_placer.cc:177] computation placer alr

# Changing config file

In [8]:
with open('configs/config.yaml', 'w') as f:
    f.write("""
### 0. Experiment
experiment:
  name: "Experiments"
  log_patch_address: "/kaggle/working/adversarial-patch-transferability/Experiments/"
  device: "cuda"
  # Logging controls
  # Set to WARNING for minimal console output; DEBUG/INFO for more detail
  log_level: "INFO"
  # If True, suppress verbose startup banners and full config dump
  minimal_logs: True

### 1.Model
model:
  ### name: 'pidnet_s'
  name: "segformer"
  hf_name: "nvidia/segformer-b0-finetuned-cityscapes-1024-1024"
  segformer_name: 'segformer_b0'
  # If you have a local HF folder (config.json + pytorch_model.bin):
  segformer_ckpt_dir: /kaggle/input/my_local_hf_segformer_dir
  # OR if you saved an HF-style state_dict to a .pth:
  segformer_ckpt: /kaggle/input/segformer_b0_hf_style/my_state_dict.pth

### 2. Dataa
dataset:
  name: "cityscapes"
  root: "/kaggle/input/cityscapes-for-segmentation/Cityscapes/"
  train: "train.txt"
  test: "test.txt"
  val: "val.txt"
  trainval: "trainval.txt"
  num_classes: 19

other_dataset:
  name: "bdd100k"
  root: "/kaggle/input/solesensei_bdd100k/bdd100k_seg/bdd100k/seg/"
  train: "train.txt"
  test: "test.txt"
  val: "val.txt"
  trainval: "trainval.txt"
  num_classes: 19

### 3. Patch
patch:
  size: 200
  #loc: "corner"
  #loc: "random"
  loc: "center"


### 3.Optimizer
optimizer:
  optimizer: "sgd"
  ### init_lr: 0.005
  momentum: 0.9
  weight_decay: 0.0005
  nesterov: False
  exponentiallr: True
  exponentiallr_gamma: 0.995
  init_lr: 2.0e-2
  use_pgd: true          # set true for PGD
  pgd_steps: 7
  pgd_alpha: 0.007843     # 2/255

## 4. Loss
loss:
  use_ohem: True
  ohemthres: 0.9
  ohemkeep: 131072
  balance_weights: [0.4, 1.0]
  sb_weights: 1.0

  ### For Vit To CNN transfer Training
  tv_weight: 1.0e-4
  attn_hijack_w: 1.0e-1
  boundary_w: 2.0e-1
  freq_w: 5.0e-2
  grad_align_w: 1.0e-1    # needs surrogate to be enabled


### 5.Training 
train:
  width: 1024
  height: 1024
  base_size: 2048
  flip: True
  shuffle: True
  ignore_label: 255
  num_workers: 2
  pin_memory: True
  drop_last: False
  batch_size: 2     # adjust according to gpu resources
  multi_scale: True
  scale_factor: 16
  power: 2.5
  log_per_iters: 1
  start_epoch: 0
  end_epoch: 20
  max_batches_per_epoch: 150      # was ~2975
  max_epochs: 20                  # was 30 (you can still resume later)
  surrogate_every: 4              # run surrogate every 4th step only
  vit_downscale: 0.75             # feed a 0.75× resized image to SegFormer
  # finetune: False
  # finetune_add: '/content/drive/MyDrive/Colab Notebooks/1_Papers/2_RobustRealtimeSS/1_Pretraining_cityscape/1_PIDNet/experiments/pidnet_s_crop1024x1024_ExponentialLR0995_batch10_basesize2048_randscale0_5to2/checkpoints/2025-01-02 08:36:14 EST-0500_141_0.648.pth.tar'


### 6. Test
test:
  width: 2048
  height: 1024
  base_size: 1024
  batch_size: 1
  output_index_pidnet: 1 # at index 1, we have I branch output
  output_index_icnet: 0
  output_index_bisenet: 0
  num_workers: 2
  pin_memory: True
  drop_last: False
  multi_scale: False
  flip_test: False
  
### 7. attack

attack:
  gamma_start: 0.9
  gamma_end: 0.5
  beta_start: 0.1
  beta_end: 0.5
  margin: 0.1
  lambda_ent: 0.1
  
  use_margin:  False

  # new entries for schedulers & mixing
  stage1_epochs: 15        # how many epochs are “Stage 1”
  stage2_epochs: 15        # how many epochs are “Stage 2” (so total=end_epoch)
  eta:            0.5      # weight for mixing L1 vs L2 (per-pixel blend)

  # feature-space divergence
  use_feat_div:  False     # set True to turn on feature-divergence term

  # (optional) EOT curriculum flag
  use_hard_eot:  True      # if True, you can detect epoch ≥ stage1_epochs and switch to stronger EOT
  use_segpgd: true        # turn SegPGD mode on/off
  segpgd_steps: 3         # T: inner attack steps per batch (start with 2–3)
  segpgd_alpha: 0.005     # α: per-step step size for the patch update (use your epsilon if you like)

freq:
  r_low: 0.12
  r_high: 0.45

surrogate:
  enable: true
  name: "pidnet_s"
""")

# Trainer

In [9]:
# 1) Load config (uses model.hf_name / segformer_ckpt_dir / segformer_ckpt / segformer_name)
config = get_config_from_yaml('configs/config.yaml', model='segformer')
# config.experiment.log_level= "INFO"
main_logger = process_config(config)

# 2) Train patch
trainer = PatchTrainer(config, main_logger)
adv_patch, iou_log = trainer.train()

# 3) Save results
out_path = config.experiment.log_patch_address + "segformer_patch.p"
pickle.dump((adv_patch.cpu(), iou_log), open(out_path, "wb"))
print("saved to:", out_path)

Experiments | 2026-01-24 18:46:35.428364


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/15.0M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/14.9M [00:00<?, ?B/s]

[Surrogate] Loaded CNN: pidnet_s with pretrained weights
Start training | Total Epochs: 20 (Stage-1: 0–9, Stage-2(JS): 10–19) | Iterations/epoch: 1488
[Limiter] max_batches_per_epoch=150, max_epochs=20
Epoch 0: using Stage-1


...

Epoch 18/19:   0%|          | 0/150 [00:00<?, ?it/s]

-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss-------------------------

Epoch 19/19:   0%|          | 0/150 [00:00<?, ?it/s]

-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss--------------------------------
-------------JS Divergence Stage 2 loss-------------------------

saved to: /kaggle/working/adversarial-patch-transferability/Experiments/segformer_patch.p


In [10]:
# type(config)
# # type(config.experiment)

In [11]:
# config.experiment.log_level

In [12]:
# from dataset.cityscapes import Cityscapes
# from dataset.bdd100k import BDD100K

# import yaml
# config = get_config_from_yaml('configs/config.yaml')

In [13]:
# cityscape_data = Cityscapes(
#           root = config.dataset.root,
#           list_path = config.dataset.train,
#           num_classes = config.dataset.num_classes,
#           multi_scale = False,
#           flip = False,
#           ignore_label = config.train.ignore_label,
#           base_size = config.train.base_size,
#           crop_size = (config.train.height,config.train.width),
#         )

# # small_dataset = Subset(cityscape_data, range(4))

# ctdataloader = torch.utils.data.DataLoader(dataset=cityscape_data,
#                                             batch_size=4,
#                                             shuffle=True,
#                                             num_workers=config.train.num_workers,
#                                             pin_memory=config.train.pin_memory,
#                                             drop_last=config.train.drop_last)

In [14]:
# bdd_data = BDD100K(
#           root = "/kaggle/input/solesensei_bdd100k/bdd100k_seg/bdd100k/seg/",
#           list_path = config.other_dataset.val,
#           num_classes = 19,
#           use_color_labels=False,
#           multi_scale = False,
#           flip = False,
#           ignore_label = config.train.ignore_label,
#           base_size = config.train.base_size,
#           crop_size = (config.train.height,config.train.width),
#         )

# # small_dataset = Subset(bdd_data, range(4))

# dataloader = torch.utils.data.DataLoader(dataset=bdd_data,
#                                             batch_size=4,
#                                             shuffle=False,
#                                             num_workers=config.train.num_workers,
#                                             pin_memory=config.train.pin_memory,
#                                             drop_last=config.train.drop_last)

In [15]:
# for batches in ctdataloader:
#     image,label,_,_,_ =batches
#     # for i in range(4):
#         # print(torch.unique(label[i]))
#     break
    

In [16]:
# plt.imshow(label[0].cpu()*14)

In [17]:
# for batches in dataloader:
#     ctimage,ctlabel,_,_,_ =batches
#     for i in range(4):
#         print(torch.unique(ctlabel[i]))
#     break
    